# Instacart Synthetic Data Analysis

Este notebook recrea (de forma simplificada) el proyecto del Sprint‑4 usando **datos sintéticos** generados en memoria. Cada apartado muestra el ejercicio, la solución paso a paso y una explicación breve.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
plt.style.use('default')
np.random.seed(42)

## 1. Generación de datos sintéticos

A continuación se generan tablas que imitan la estructura de `orders`, `order_products`, `products`, `departments` y `aisles` del conjunto *Instacart* original.

In [ ]:

# ----- Parámetros básicos -----
n_users = 5000
n_orders = 50000
n_products = 200
n_departments = 10
n_aisles = 20

# Departamentos y pasillos ficticios
departments = pd.DataFrame({
    "department_id": range(1, n_departments+1),
    "department": [f"Dept_{i}" for i in range(1, n_departments+1)]
})

aisles = pd.DataFrame({
    "aisle_id": range(1, n_aisles+1),
    "aisle": [f"Aisle_{i}" for i in range(1, n_aisles+1)]
})

# Productos
products = pd.DataFrame({
    "product_id": range(1, n_products+1),
    "product_name": [f"Product_{i}" for i in range(1, n_products+1)],
    "aisle_id": np.random.choice(aisles["aisle_id"], n_products),
    "department_id": np.random.choice(departments["department_id"], n_products)
})

# Órdenes
orders = pd.DataFrame({
    "order_id": range(1, n_orders+1),
    "user_id": np.random.choice(range(1, n_users+1), n_orders),
    "order_number": np.random.randint(1, 101, n_orders),
    "order_dow": np.random.randint(0, 7, n_orders),
    "order_hour_of_day": np.random.randint(0, 24, n_orders),
    "days_since_prior_order": np.random.choice([np.nan]+list(range(1, 31)), n_orders, p=[0.05]+[0.95/30]*30)
})

# order_products: para cada order_id añadir 1‑15 productos
rows=[]
for oid in orders["order_id"]:
    n_items=np.random.randint(1,16)
    prods=np.random.choice(products["product_id"], n_items)
    for pos, pid in enumerate(prods, start=1):
        rows.append({
            "order_id": oid,
            "product_id": pid,
            "add_to_cart_order": pos,
            "reordered": np.random.binomial(1, 0.4)  # 40 % prob.
        })
order_products = pd.DataFrame(rows)


## Vista rápida de las tablas
orders.head(), order_products.head(), products.head()

In [ ]:
orders.head()

In [ ]:
order_products.head()

In [ ]:
products.head()

## Un poco de exploración de datos

## 2. [A] Fácil

Realicemos **tres** ejercicios introductorios.

### A1. Verificar rangos de `order_hour_of_day` (0‑23) y `order_dow` (0‑6)

In [ ]:

hour_ok = orders["order_hour_of_day"].between(0,23).all()
dow_ok = orders["order_dow"].between(0,6).all()
print(f"order_hour_of_day válido 0‑23 → {hour_ok}")
print(f"order_dow válido 0‑6 → {dow_ok}")


Ambas columnas cumplen sus rangos, lo que indica que la generación sintética respeta los valores esperados.

### A2. Número de órdenes por hora del día

In [ ]:

orders_per_hour = orders.groupby("order_hour_of_day")["order_id"].nunique()
orders_per_hour.plot(kind="bar", figsize=(10,4))
plt.title("Órdenes por hora del día")
plt.xlabel("Hora")
plt.ylabel("Número de órdenes")
plt.tight_layout()
plt.show()


Observamos un patrón simulado (prácticamente uniforme) porque usamos una distribución uniforme; en datos reales suelen destacarse las horas intermedias del día.

### A3. Órdenes por día de la semana

In [ ]:

orders_per_dow = orders.groupby("order_dow")["order_id"].nunique()
orders_per_dow.plot(kind="bar", figsize=(8,4))
plt.title("Órdenes por día de la semana (0=Lunes)")
plt.xlabel("Día de la semana")
plt.ylabel("Número de órdenes")
plt.tight_layout()
plt.show()


De nuevo vemos una ligera variación por azar. Ajustando la distribución se podrían imitar picos de fin de semana.

## 3. [B] Intermedio

Solo resolveremos **dos** ejercicios representativos.

### B1. Top‑10 productos más vendidos

In [ ]:

top10 = (order_products
         .groupby("product_id")["order_id"]
         .nunique()
         .sort_values(ascending=False)
         .head(10)
         .reset_index())
top10 = top10.merge(products[["product_id","product_name"]], on="product_id")
top10


Utilizamos `groupby('product_id').nunique()` para contar órdenes únicas por producto de manera concisa.

### B2. Ratio de reorden global

In [ ]:

reorder_ratio = order_products["reordered"].mean()
print(f"Reorder ratio global: {reorder_ratio:.2%}")


El *reorder ratio* indica la proporción de ítems comprados que ya habían sido adquiridos previamente por la persona usuaria (40 % en promedio, según la probabilidad definida).

## 4. [C] Difícil

Seleccionamos **dos** retos analíticos avanzados.

### C1. Departamento con mayor *reorder ratio*

In [ ]:

# Enlazar tablas
merged = order_products.merge(products, on="product_id").merge(departments, on="department_id")
dept_reorder = merged.groupby("department")["reordered"].mean().sort_values(ascending=False)
dept_reorder.head(5)


Filtramos los 5 departamentos con mayor proporción de re-compra; el cálculo se simplifica con un `groupby` directo.

### C2. Distribución de días entre pedidos por usuaria/o

In [ ]:

# Para cada usuario, calcular el promedio de días entre pedidos (excluyendo NaN inicial)
user_gap = orders.groupby("user_id")["days_since_prior_order"].mean()
user_gap.hist(bins=30, figsize=(8,4))
plt.title("Media de días entre pedidos por usuario/a")
plt.xlabel("Días")
plt.ylabel("Frecuencia")
plt.tight_layout()
plt.show()


Se observa una distribución aproximadamente uniforme entre 1 y 30 días debido al diseño del simulador; en datos reales suele observarse una cola a la derecha.

## 5. Conclusiones

Con datos totalmente sintéticos se replicó, de forma reducida, el análisis del Sprint‑4. El enfoque muestra cómo validar rangos de datos, explorar patrones temporales y evaluar métricas como el *reorder ratio* usando operaciones simples de `groupby`.